# 02 — Exploración de conjuntos de datos (funcionalidad 010, tarea T25)

Revisa **cada conjunto candidato** antes de armar la receta v1, sin bajar ningún conjunto entero:

| Qué verifica | Por qué |
|---|---|
| Disponibilidad y **revisión exacta** (commit del conjunto) | La receta fija una revisión: la licencia puede cambiar entre revisiones |
| **Licencia** declarada en la ficha | Sin licencia verificable, el conjunto queda **excluido** (CA2) |
| Columnas | Confirmar los nombres que usa la receta |
| Imágenes por clase y por **generador** | Elegir menos generadores con más imágenes cada uno |
| Formato, tamaño y calidad JPEG por clase | Ver si hay sesgo de formato («Fake or JPEG?») |

Los candidatos están en `red_neuronal/configuracion/candidatos_v1.yaml`. Toda la lógica está en el paquete (`python -m clasificador explorar`); el cuaderno solo la llama y muestra los resultados.

**Antes de ejecutar:**
1. Con la **misma cuenta** del `HF_TOKEN`, entrar a la página de cada conjunto que pida aceptar condiciones y aceptarlas (OpenFake, Community Forensics, CarDD, ConstructionSite). Si no, aparece como «no disponible» con un error 401 o 403.
2. Basta con **CPU** (*Entorno de ejecución → Cambiar tipo → CPU*): no usa GPU y no gasta unidades de cómputo. Tarda unos 10 a 20 minutos.
3. *Entorno de ejecución → Ejecutar todas*.

## 1. Parámetros y Google Drive

In [ ]:
REVISION = "funcionalidad/010-red-neuronal"   # rama o commit del repositorio

REPOSITORIO = "valentinasalmon/PFG-sistema-deteccion-IA"
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
TRABAJO = Path("/content")
CODIGO = TRABAJO / "codigo"

RESPALDO = Path("/content/drive/MyDrive/evidentia/exploracion")
RESPALDO.mkdir(parents=True, exist_ok=True)
print("Resultados en:", RESPALDO)

## 2. Credenciales y código

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
_token_github = userdata.get("GITHUB_TOKEN")
if not os.environ["HF_TOKEN"] or not _token_github:
    raise RuntimeError("Falta HF_TOKEN o GITHUB_TOKEN en los secretos (ícono de la llave), "
                       "o están vacíos. Cargalos con Acceso al cuaderno activado y volvé a ejecutar todo.")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
print("Credenciales cargadas (no se muestran).")

import base64
import subprocess


def git_seguro(*argumentos, token=None):
    """Ejecuta git sin que el token aparezca nunca en el comando ni en los errores.

    El token viaja en una variable de entorno de configuración de git
    (cabecera de autorización), no en la URL. Si git falla, se muestra solo
    su mensaje de error, que no contiene el token.
    """
    entorno = dict(os.environ)
    if token:
        credencial = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        entorno.update({
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credencial}",
            "GIT_TERMINAL_PROMPT": "0",
        })
    proceso = subprocess.run(["git", *argumentos], env=entorno, capture_output=True, text=True)
    if proceso.returncode != 0:
        mensaje = proceso.stderr.replace(token, "***") if token else proceso.stderr
        if "could not read Username" in mensaje or "Authentication failed" in mensaje:
            mensaje += (
                "\n\nGitHub rechazó el GITHUB_TOKEN: venció, está mal copiado o no tiene acceso "
                "a este repositorio. Revisá en GitHub (Settings → Developer settings → "
                "Fine-grained tokens) que tenga el repositorio PFG-sistema-deteccion-IA y "
                "Contents: Read-only; si dudás, creá uno nuevo, reemplazá el secreto y "
                "ejecutá todo otra vez."
            )
        raise RuntimeError(f"git {argumentos[0]} falló:\n{mensaje.strip()}") from None
    return proceso.stdout.strip()


if CODIGO.exists():
    subprocess.run(["rm", "-rf", str(CODIGO)], check=True)
git_seguro("clone", "--quiet", f"https://github.com/{REPOSITORIO}.git", str(CODIGO),
           token=_token_github)
git_seguro("-C", str(CODIGO), "fetch", "--quiet", "origin", REVISION, token=_token_github)
git_seguro("-C", str(CODIGO), "checkout", "--quiet", REVISION)
del _token_github

commit = git_seguro("-C", str(CODIGO), "rev-parse", "HEAD")
print("Commit en uso:", commit)

In [ ]:
!pip install -q -e "{CODIGO}/red_neuronal[entrenamiento]"
!python -m clasificador explorar --ayuda
import sys
sys.path.insert(0, str(CODIGO / "red_neuronal"))  # el kernel ya estaba abierto al instalar

## 3. Explorar los candidatos

Por cada conjunto se ve `Explorando ...`. Un conjunto que falla no detiene a los demás: queda excluido con su motivo.

In [ ]:
import subprocess, time

def correr(*argumentos):
    """Ejecuta el paquete y muestra su salida línea por línea (Colab no la muestra sola)."""
    proceso = subprocess.Popen(
        ["python", "-m", "clasificador", *argumentos],
        cwd=CODIGO / "red_neuronal", text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for linea in proceso.stdout:
        print(linea, end="")
    return proceso.wait()

inicio = time.perf_counter()
codigo_salida = correr("explorar", "--candidatos", "configuracion/candidatos_v1.yaml",
                       "--salida", str(RESPALDO))
print(f"--> explorar: {time.perf_counter() - inicio:.0f} s (código {codigo_salida})")

## 4. Tabla de fuentes con su licencia

Esta es la tabla que pide la tarea T25. **Copiala al chat** tal cual.

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RESPALDO / "exploracion.md").read_text()))

## 5. Detalle de cada conjunto: columnas, generadores y formatos

In [ ]:
import json
informes = json.loads((RESPALDO / "exploracion.json").read_text())
for informe in informes:
    print("=" * 80)
    print(informe["conjunto"], "→", "INCLUIDO" if informe["incluido"] else "EXCLUIDO")
    for motivo in informe["motivos_exclusion"]:
        print("   motivo:", motivo)
    print("   revisión exacta:", informe["revision_exacta"] or "—")
    print("   licencia:", informe["licencia"] or "—",
          "| pide aceptar condiciones:", informe["requiere_aceptar_condiciones"])
    print("   archivos Parquet:", informe["cantidad_archivos"], "| columnas:", informe["columnas"])
    print("   en la muestra:", informe["por_clase"])
    generadores = list(informe["por_generador"].items())
    if generadores:
        print(f"   {len(generadores)} generadores; los 15 con más imágenes:")
        for nombre, cantidad in generadores[:15]:
            print(f"      {cantidad:6d}  {nombre}")
    for clase, resumen in informe["resumen_formatos"].items():
        print(f"   formato {clase}: {resumen}")

## 6. Ejemplos visuales

Unas imágenes de cada conjunto incluido, para ver con los propios ojos qué contiene (por ejemplo, si las «reales» son de verdad fotos y no ilustraciones).

In [ ]:
import io
import matplotlib.pyplot as plt
from PIL import Image
from clasificador.datos.exploracion import InspectorHuggingFace, cargar_candidatos

inspector = InspectorHuggingFace()
candidatos = {c.conjunto: c for c in cargar_candidatos(
    CODIGO / "red_neuronal/configuracion/candidatos_v1.yaml").candidatos}
for informe in informes:
    if not informe["incluido"]:
        continue
    candidato = candidatos[informe["conjunto"]]
    archivo = next(iter(informe["archivos_explorados"]))
    columnas = [c for c in (candidato.columna_imagen, candidato.columna_clase) if c]
    filas = inspector.leer_imagenes(candidato.conjunto, informe["revision_exacta"],
                                    archivo, columnas, 6)
    figura, ejes = plt.subplots(1, len(filas), figsize=(3 * len(filas), 3))
    for eje, fila in zip(ejes if len(filas) > 1 else [ejes], filas):
        valor = fila[candidato.columna_imagen]
        eje.imshow(Image.open(io.BytesIO(valor["bytes"] if isinstance(valor, dict) else valor)))
        eje.set_title(str(fila.get(candidato.columna_clase, candidato.clase_fija)), fontsize=8)
        eje.axis("off")
    figura.suptitle(informe["conjunto"])
    plt.show()

## 7. Propuesta de receta v1 (tarea T27)

Arma, para las fuentes de entrenamiento incluidas, el bloque de la receta: **revisión fija**, **archivos elegidos al azar** con la semilla, los **generadores con más imágenes** y una **cuota por generador**. Se guarda en Drive como `propuesta_receta_v1.yaml`. Copiá su contenido al chat: con eso el agente completa `configuracion/v1.yaml`.

In [ ]:
import yaml
from clasificador.datos.exploracion import InformeConjunto, proponer_fuente

# Cantidades pedidas por fuente: (reales, sintéticas, cantidad de generadores).
PEDIDOS = {
    "community_forensics": (20000, 20000, 50),
    "openfake": (15000, 15000, 10),
}
propuestas = []
for informe in informes:
    if informe["incluido"] and informe["conjunto"] in candidatos:
        nombre = candidatos[informe["conjunto"]].nombre
        if nombre in PEDIDOS:
            datos = {k: v for k, v in informe.items() if k != "resumen_formatos"}
            propuestas.append(proponer_fuente(InformeConjunto(**datos),
                                              candidatos[informe["conjunto"]],
                                              **dict(zip(("cantidad_reales", "cantidad_sinteticas", "cantidad_generadores"), PEDIDOS[nombre])), semilla=42))
texto = yaml.safe_dump({"fuentes": propuestas}, allow_unicode=True, sort_keys=False)
(RESPALDO / "propuesta_receta_v1.yaml").write_text(texto)
print(texto)

## 8. Qué pasar al chat

1. La **tabla** de la sección 4.
2. La salida de la sección 5 (columnas y generadores).
3. La **propuesta** de la sección 7.

Antes de pegar, revisar que no aparezca ningún token (`hf_…`, `github_pat_…`). Nada de esto contiene imágenes ni datos personales.